<a href="https://colab.research.google.com/github/CMDDclass/MS697-material/blob/main/Hands-on-session4/1_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<br><br>
<font size='6'><b>AI-driven Materials Development</b></font><br><br>

This notebook was prepared with reference to the Machine Learning course materials by Seungchul Lee (KAIST).

In this notebook, we'll use a GPU for faster model training.

To enable the GPU:

1. Navigate to Runtime.

2. Select Change runtime type.

3. Choose T4 GPU from the dropdown menu.

4. Click Save.

# Generative Adversarial Networks (GAN)



1. From Discriminative to Generative Models
	•	Discriminative Model: learns the boundary between classes
→ models $P(y|x)$ (e.g., logistic regression, CNN classifier)
	•	Generative Model: learns to generate data itself
→ models $P(x)$ or joint $P(x, y)$
→ can sample new data resembling the real one



2. Density Function Estimation
	•	Real-world data (e.g., images) can be seen as points in a high-dimensional space
→ e.g., a $64 \times 64 \times 3$ image = one point in $\mathbb{R}^{12288}$
	•	Goal: estimate a model distribution $P_{\text{model}}(x)$ close to real $P_{\text{data}}(x)$
→ if successful, we can generate new samples by drawing from $P_{\text{model}}(x)$
	•	Distance between two distributions can be measured by Kullback–Leibler (KL) divergence
	•	Neural networks can learn a deterministic transformation:
$z \sim N(0, I) \rightarrow x = G(z)$
where $G$ is a generator network mapping latent code $z$ to data $x$



3. GAN Framework
	•	GANs do not explicitly estimate $P_{\text{model}}(x)$
→ instead, use a game-theoretic approach
	•	Two networks:
	•	Generator (G): tries to create realistic samples $x = G(z)$
	•	Discriminator (D): tries to distinguish real data from fake samples
	•	Objective:
	•	$G$ learns to fool $D$
	•	$D$ learns to detect fakes



4. Objective Function

Discriminator Loss

$\max_D ; E_{x \sim p_{\text{data}}(x)}[\log D(x)] + E_{z \sim p_z(z)}[\log(1 - D(G(z)))]$

Generator Loss (Non-saturating)

$\max_G ; E_{z \sim p_z(z)}[\log D(G(z))]$

→ gives stronger gradients early in training



5. Training as a Min–Max Game
	•	Alternating optimization:
	1.	Fix G, update D to improve real/fake classification
	2.	Fix D, update G to make better fakes
	•	The equilibrium is reached when $P_G(x) \approx P_{\text{data}}(x)$
→ Discriminator cannot tell real from fake (outputs ≈ 0.5)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model

In [ ]:
!nvidia-smi
tf.config.list_physical_devices("GPU")

In [ ]:
LATENT_DIM = 100
BATCH_SIZE = 128
LABEL_SMOOTH = 0.9
D_STEPS = 2
N_ITER = 4000
LR = 2e-4
BETA_1 = 0.5

In [ ]:
def make_noise(samples):
    return np.random.normal(0, 1, [samples, 100])

def plot_generated_images(generator, samples=16):
    noise = make_noise(samples)
    generated_images = generator.predict(noise)
    generated_images = generated_images.reshape(samples, 28, 28)

    plt.figure(figsize=(6, 6))
    for i in range(samples):
        plt.subplot(4, 4, i + 1)
        plt.imshow(generated_images[i], cmap='gray', interpolation='nearest')
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def show_grid(imgs, grid=(4,4), fname=None, title=None, image_size=28):
    imgs = np.asarray(imgs)

    # (N, H, W, 1) → (N, H, W)
    if imgs.ndim == 4 and imgs.shape[-1] == 1:
        imgs = imgs[..., 0]

    # (N, 784) → (N, 28,28)
    if imgs.ndim == 2 and imgs.shape[1] == image_size * image_size:
        imgs = imgs.reshape(-1, image_size, image_size)

    if imgs.ndim != 3:
        raise ValueError(f"Expect (N,H,W) or (N,784), got {imgs.shape}")

    r, c = grid
    canvas = np.vstack([
        np.hstack([imgs[i*c + j] for j in range(c)])
        for i in range(r)
    ])

    plt.figure(figsize=(c, r))
    plt.axis('off')
    if title: plt.title(title)
    plt.imshow(canvas, cmap='gray', vmin=0, vmax=1)
    if fname:
        plt.savefig(fname, bbox_inches='tight', pad_inches=0)
    plt.show()
(train_x, train_y), _ = tf.keras.datasets.mnist.load_data()

print('train_images(0~9)', train_x.shape)

train_x = train_x[np.where(train_y == 2)]
train_x = train_x/255.0
train_x = train_x.reshape(-1, 784)


print('train_iamges(2) :', train_x.shape)
show_grid(train_x[:16], grid=(4,4), title="MNIST Digit 2 Samples")


train_dataset = tf.data.Dataset.from_tensor_slices(train_x) \
                               .shuffle(train_x.shape[0]) \
                               .batch(BATCH_SIZE, drop_remainder=True) \
                               .repeat()
train_iter = iter(train_dataset)

In [ ]:
def build_generator():
    z = layers.Input(shape=(LATENT_DIM,))
    x = layers.Dense(256, use_bias=False)(z)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dense(784, activation='tanh')(x)
    return models.Model(z, x, name="generator")
generator = build_generator()
generator.summary()

In [ ]:
def build_discriminator():
    x_in = layers.Input(shape=(784,))
    x = layers.Dense(512)(x_in)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    logits = layers.Dense(1)(x) # No activation (from_logits=True)
    return models.Model(x_in, logits, name="discriminator")
discriminator = build_discriminator()
discriminator.summary()

In [ ]:
plot_model(
    generator,
    show_shapes=True,
    show_layer_names=True,
    rankdir="TB",
    dpi=160
)


In [ ]:
plot_model(
    discriminator,
    show_shapes=True,
    show_layer_names=True,
    rankdir="TB",
    dpi=160
)


Step 1: Fix $G$ and perform a gradient step to

<br>

$$\min_{D} E_{x \sim p_{\text{data}}(x)}\left[-\log D(x)\right]  + E_{x \sim p_{z}(z)}\left[-\log (1-D(G(z)))\right]$$

<br>

Step 2: Fix $D$ and perform a gradient step to

<br>

$$\min_{G} E_{x \sim p_{z}(z)}\left[-\log D(G(z))\right]$$

In [ ]:
bce_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
d_optimizer = tf.keras.optimizers.Adam(learning_rate=LR, beta_1=BETA_1)
g_optimizer = tf.keras.optimizers.Adam(learning_rate=LR, beta_1=BETA_1)

In [ ]:
real_labels_smoothed = tf.ones([BATCH_SIZE, 1], dtype=tf.float32) * LABEL_SMOOTH
fake_labels = tf.zeros([BATCH_SIZE, 1], dtype=tf.float32)
g_target_labels = real_labels_smoothed

In [ ]:
@tf.function
def train_step_d(real_images):
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as tape:
        generated_images = generator(noise, training=True)

        real_logits = discriminator(real_images, training=True)
        fake_logits = discriminator(generated_images, training=True)

        d_loss_real = bce_loss(real_labels_smoothed, real_logits)
        d_loss_fake = bce_loss(fake_labels, fake_logits)
        d_loss = d_loss_real + d_loss_fake

    grads = tape.gradient(d_loss, discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))
    return d_loss

In [ ]:
@tf.function
def train_step_g():
    noise = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as tape:
        generated_images = generator(noise, training=True)
        combined_logits = discriminator(generated_images, training=False)

        g_loss = bce_loss(g_target_labels, combined_logits)

    grads = tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
    return g_loss

In [ ]:
print("Training starts...")
for i in range(N_ITER):

    d_loss_avg = 0.0
    for _ in range(D_STEPS):
        real_images_batch = next(train_iter)
        d_loss = train_step_d(real_images_batch)
        d_loss_avg += d_loss

    g_loss = train_step_g()

    if i % 1000 == 0:
        d_loss_val = (d_loss_avg / D_STEPS).numpy()
        g_loss_val = g_loss.numpy()
        print(f"iter {i} | D: {d_loss_val:.4f} | G: {g_loss_val:.4f}")
        plot_generated_images(generator)


# After Training


- After training, use the generator network to generate new data



In [ ]:
plot_generated_images(generator)

# Practice:  Generate a Specific Digit ("9") with a Standard GAN

Train a generator using the MNIST dataset and generate images of the digit 9.

### **Submit the following as your practice result:
• Generated "9" images (9 images)